# Оценка сложности. 
1. Что такое сложность
2. Либо задача регрессии (предсказать уровень читаемости) либо задача ранжирования
3. Структурные признаки текста

Загрузка данных


In [1]:
import os


folder1 = r"IMS_extracted\IMS2013-2024\IMS2023"
folder2 = r"IMS_extracted\IMS2013-2024\IMS2024"


def load_full_articles(folder_path):
    # Берем только .txt файлы с полными текстами (без Abstract и KW)
    txt_files = [
        file for file in os.listdir(folder_path)
        if file.endswith('_rus.txt') and 'Abstract' not in file and 'KW' not in file
    ]

    texts = []
    for filename in txt_files:
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            texts.append(file.read().strip())

    # Объединяем в одну строку
    return "\n\n".join(texts)

In [2]:
text1 = load_full_articles(folder1)
text2 = load_full_articles(folder2)

# Объединяем
combined_text = text1 + "\n\n" + text2

# Texstat
Работает на основе количественных признаков (кол-во слов, слогов, предложений и т.д.).
Реализует классические метрики:
Flesch Reading Ease,
Flesch–Kincaid Grade,
Gunning Fog Index,
SMOG,
ARI
Dale–Chall,
Coleman–Liau и др.

In [17]:
! pip install textstat

^C


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import textstat

def analyze_text_readability(text):
    metrics = {
        "words": textstat.lexicon_count(text, removepunct=True), #Количество слов
        "sentences": textstat.sentence_count(text), #средняя длина предложения
        "syllables": textstat.syllable_count(text), #кол-во слогов нужно для формул
        "flesch_reading_ease": textstat.flesch_reading_ease(text), #Оценивает читаемость текста по шкале от 0 до 100.
        # Чем выше значение, тем проще текст. Основана на длине предложений и количестве слогов.
        # Надёжна только для английского языка.
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(text),#Основана на той же формуле, что и Flesch Reading Ease, 
        #но интерпретируется как "школьный уровень".
        "gunning_fog": textstat.gunning_fog(text),
        "smog_index": textstat.smog_index(text),# даёт оценку уровня образования, необходимого для понимания текста.
        "coleman_liau_index": textstat.coleman_liau_index(text), #Читаемость, основанная на количестве букв в словах и длине предложений
        #>14 — академический стиль
        "automated_readability_index": textstat.automated_readability_index(text),#Индекс читаемости на основе длины слов и предложений
        #Значение ~8 — читаемо, >12 — высокий уровень сложности.
        "dale_chall_score": textstat.dale_chall_readability_score(text), #Сложность лексики по списку частотных слов
        "linsear_write_formula": textstat.linsear_write_formula(text),#Оценка по военной шкале читаемости
        "difficult_words": textstat.difficult_words(text),#Количество редких или сложных слов(eng)
        "reading_time_minutes": textstat.reading_time(text),#	Примерное время чтения текста
        "text_standard": textstat.text_standard(text, float_output=False) #Средний "школьный класс" по совокупным метрикам
    }

    df = pd.DataFrame(metrics.items(), columns=["Метрика", "Значение"])
    return df


In [ ]:
df = analyze_text_readability(combined_text)
print(df.to_string(index=False))

                    Метрика          Значение
                      words             13729
                  sentences               777
                  syllables             14087
        flesch_reading_ease            104.27
       flesch_kincaid_grade               3.1
                gunning_fog              7.24
                 smog_index               5.2
         coleman_liau_index             22.39
automated_readability_index              21.2
           dale_chall_score              9.76
      linsear_write_formula          6.142857
            difficult_words               104
       reading_time_minutes           1446.44
              text_standard 5th and 6th grade


Метрики, не зависящие от слогов и словарей:
coleman_liau_index, automated_readability_index, gunning_fog

# Ruts
Поддерживает метрики, адаптированные под русскую морфологию и структуру. Позволяет вычислять для текста следующие метрики удобочитаемости:

Тест Флеша-Кинкайда (Уровень образования (в годах обучения), необходимый для понимания текста.)

Индекс удобочитаемости Флеша (0–100-балльная шкала. Чем выше, тем проще читать текст.)

Индекс Колман-Лиау (Уровень читаемости на основе длины слов (в символах) и предложений.)

Индекс SMOG (Количество лет образования, нужное для понимания (основывается на сложных словах))

Автоматический индекс удобочитаемости (Учитывает длину слов и предложений. Больше значение — выше сложность.)

Индекс удобочитаемости LIX (Шведский индекс читаемости. Учитывает длину предложений и долю длинных слов.)

Использует токенизацию, морфоанализ, словоформы и возможно — частотные словари.

In [ ]:
! pip install ruts

In [ ]:
! pip install --upgrade ruts


In [10]:
from ruts import BasicStats
text = combined_text
bs = BasicStats(text)
bs.print_stats()

     Статистика     | Значение 
------------------------------
Предложения         |   718    
Слова               |  13951   
Уникальные слова    |   4659   
Длинные слова       |   8623   
Сложные слова       |   4521   
Простые слова       |   7809   
Односложные слова   |   2220   
Многосложные слова  |  10110   
Символы             |  116801  
Буквы               |  91050   
Пробелы             |  18337   
Слоги               |  38530   
Знаки препинания    |   3882   


In [8]:
from ruts import ReadabilityStats
text = combined_text
rs = ReadabilityStats(text)
rs.print_stats()

                Метрика                 | Значение 
--------------------------------------------------
Тест Флеша-Кинкайда                     |  13.09   
Индекс удобочитаемости Флеша            |  15.59   
Индекс Колман-Лиау                      |  13.76   
Индекс SMOG                             |  22.24   
Автоматический индекс удобочитаемости   |  15.27   
Индекс удобочитаемости LIX              |  81.24   
